# Introduction to Machine Learning

The aim of this notebooks is to introduce students to some fundaments of Machine Learning. For it, we will study a commonly known 

Across this notebook, we will see the following contents:

1. Loading Tabular Data, Exploratory Data Analysis (EDA) and Handling Null Values.

2. Train/Validation/Test Split, Evaluation Metrics.

3. Preprocessing.

4. Baseline Model: Logistic Regression with Tensorflow.

5. Neural Network Model in Tensorflow.

6. Hyperparameter Optimization (Optuna / GridSearch).

## 1. Loading Tabular Data, Exploratory Data Analysis (EDA) and Handling Null Values

The first step is to load the data, and perform EDA. EDA is useful to understand the characteristics of the data, identify possible predictors, and spot possible problems.

Across this notebook, we will mainly use the library Pandas for data handling. Numpy can also be used, but Pandas offers a better visualization for tabular data.

Pandas: [Webpage](https://pandas.pydata.org/docs/)

Numpy: [Webpage](https://numpy.org/doc/stable/)

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
train.replace(" ", "", inplace=True)

Now, we can visualize some parts of the data using the head method of dataframe.

In [ ]:
display(train.head())

As it can be seen, we have a set of features, each with their own values and datatypes. Some features like *age* display a numerical datatype. Others present string values, like *workclass*. To see more, lets get a description of train.csv (Comment: If you do not add "include='all'" within the options, only features with a numerical datatype will appear.)

In [ ]:
display(train.describe(include='all'))

Above, you can see multiple statistics of each feature within the dataset. Some features present information regarding their *mean* value, and their *standard deviation*. Then, features with categorical values seem to be missing that information, while presenting other information like number of unique values and frequency of the most present class. The missing information is represented by NaN (Not a Number), which is how Pandas indicates a missing value.

The next step would be to analyze the number of missing values within each feature.

In [ ]:
display(train.isnull().sum())

At first glance, it appears that there are no missing values within the dataset. Therefore, we can continue with EDA. If there were any missing values, we must handle them before training a model, as they cannot be processed by the ML models.

Furthermore, EDA can also retrieve information about additional qualities of the features, like correlations among features (multicollinearity), a better visualization of the values within a feature, or even errors in data.

The remaining parts of the EDA are left as a set of exercises. These are the following:

1. When selecting a column in pandas with *dataframe\[name_of_column\].values_count()*, you can see the unique values within the column, plus their frequency. Try for the categoricals columns within pandas. Do you spot something strange in any of them?

2. Plotly is a graphical library, which can be used for data plotting. A quite useful plot is scatter_matrix, as it presents how two values correlate between each other. This correlation are presented in a matrix of scatter plots, where the diagonal represents a scatter plot of each feature with itself, and each non-diagonal plot is a scatter plot p<sub>ij</sub>, for index i and j of list of columns of the dataframe. Try with different columns within the plot. Do you notice any strange correlation between two features?

3. Replacing values can be done by indexing the specific cells, and applying a value to them. Also, you can fill null values manually (in the preprocessing part, we will learn how to handle nulls in other ways). Can you spot any value that should be replaced by others within the dataset?

In [ ]:
# Example of Exercise 1
display(train["sex"].value_counts())

In [ ]:
# Example of exercise 2

import plotly.express as px
fig = px.scatter_matrix(train[["age", "fnlwgt"]])
fig.show()

In [ ]:
# Example of exercise 3

# Across all the dataset
train[train == 0] = 0

# Across one column
train[train["age"] == 0] = 0

# To replace nulls
train = train.fillna(0)

## 2. Train/Validation/Test Split & Evaluation Metrics

ML models learn patterns within the data which they are trained with. The idea is then that these patterns are extrapolable to real data that comes into the system, usually named as real data or inference data. For this reason, we must evaluate how well models extrapolate towards data they have not previously seen.

This is the goal of splitting the data in Training, Validation and Tests datasets. We use the Training dataset for training, the validation dataset for evaluating the model during training, and finally the Test dataset for the evaluation after the training.

For splitting the data, we can use the function *train_test_split* after the label has been separated from the dataset.

In [ ]:
from sklearn.model_selection import train_test_split

y = train.pop("label")
X = train

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

We can also prepare the testing data by reading the test dataset, and dividing it between input data and labels.

In [ ]:
test = pd.read_csv("test.csv")
test.replace(" ", "", inplace=True)

y_test = test.pop("label")
X_test = test

# Remember to handle possible null values, if they are present in the test data.

# Task: Write code here to handle the null values, if they are present.

For the evaluation, we will use the following metrics:

| Metric | Formula | Intuition |
|---|---|---|
| **Cross-Entropy Loss** | $-\frac{1}{n}\sum y \log(\hat{p}) + (1-y)\log(1-\hat{p})$ | Penalizes confident wrong predictions heavily. Lower = better. |
| **Accuracy** | $\frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$ | Fraction of correct predictions. Misleading on imbalanced classes. |
| **F1 Score** | $2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Harmonic mean of precision and recall. Robust to class imbalance. |

The code can be found below.


<!-- | **AUC-ROC** | No good function defined | Area under the ROC curve. 1.0 = perfect, 0.5 = random classifier. | -->

In [ ]:
from sklearn.metrics import log_loss, accuracy_score, f1_score

def return_labels(y_true_argmaxed, labels):
    # y_true_argmaxed = np.reshape(y_true_argmaxed, newshape=(-1, 1))
    # print(np.apply_along_axis(lambda x: print(x), 0, y_true_argmaxed))
    return [labels[selected_class] for selected_class in y_true_argmaxed]

def cross_entropy_loss(y_test, y_pred_proba, labels):
    # log(INFO, "CE Loss")
    ground_truth_np = return_labels(np.argmax(y_test, axis=1), labels)
    return log_loss(ground_truth_np, y_pred_proba, labels=labels)


def accuracy(y_test, y_pred, labels):
    # log(INFO, "Accuracy")
    y_pred = np.argmax(y_pred, axis=1)
    ground_truth_np = np.argmax(y_test, axis=1)
    return accuracy_score(ground_truth_np, y_pred)


def f1_score_local(y_test, y_pred_proba, labels):
    # log(INFO, "F1 Score")
    ground_truth_np = return_labels(np.argmax(y_test, axis=1), labels)
    predictions_np = return_labels(np.argmax(y_pred_proba, axis=1), labels)
    return f1_score(ground_truth_np, predictions_np, labels=labels, average=None, zero_division=0)

## 3. Preprocessing.

For tabular data, preprocessing can be fundamental, not only to make the code run, but also to achieve good performance on training. The following functions are provided to the students to run the preprocessing:

<u>Data Imputation functions</u>

Functions to handle missing values (`NaN`) in a DataFrame column.

| Function | Strategy | When to use |
|---|---|---|
| `constant_input(X, col, value=0)` | Replaces `NaN` with a fixed constant | When missing means a known default (e.g. 0 purchases) |
| `mean_input(X, col)` | Replaces `NaN` with the column mean (computed on `X_train`) | Numerical columns with roughly symmetric distributions |
| `most_frequent_input(X, col)` | Replaces `NaN` with the most common value (computed on `X_train`) | Categorical or skewed numerical columns |
| `remove_row(X, col)` | Drops rows where the column is `NaN` | When missing data is rare and not systematic |

<br>

> Warning `mean_input` and `most_frequent_input` fit on `X_train` and apply to any split — this prevents **data leakage**.

---

<u>Categorical Encoding</u>

Functions to convert categorical string columns into numerical representations.

| Function | Strategy | Output | When to use |
|---|---|---|---|
| `one_hot_encode_labels(y, categories)` | One-hot encodes a target Series | 1 if y == category, else 0 — one column per category | Multi-class targets (e.g. softmax output) |
| `one_hot_encode_column(X, col, categories)` | One-hot encodes a feature column, drops original | k binary columns for k categories | Nominal features with no natural order |
| `label_encode_column(X, col, categories)` | Assigns an integer to each category | 0, 1, ..., k-1 | Ordinal features with a meaningful order |

<br>

> Warning: **Label encoding implies order.** Only use it when the categories have a meaningful ranking (e.g. `low < medium < high`). For unordered categories, prefer one-hot encoding.

---

<u>Numerical Scaling</u>

Functions to rescale numerical columns. Always fit scaling parameters on `X_train` only.

| Function | Formula | Output range | When to use |
|---|---|---|---|
| `min_max_scale_column(X, col)` | (x - x_min) / (x_max - x_min) | [0, 1] | When you need bounded outputs (e.g. neural networks) |
| `std_scale_column(X, col)` | (x - mean) / std | mean=0, std=1 | When the column is roughly Gaussian; robust to outliers |

In [ ]:
from typing import Union

"""Functions for data inputing"""

def constant_input(X_dataframe: pd.DataFrame, column_name: str, input: Union[str, int, float]=0) -> pd.DataFrame:
    X_dataframe[column_name] = X_dataframe[column_name].fillna(input)
    return X_dataframe


def mean_input(X_dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    mean_value = X_train[column_name].mean()
    X_dataframe[column_name] = X_dataframe[column_name].fillna(mean_value)
    return X_dataframe


def most_frequent(X_dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    most_frequent_value = X_train[column_name].value_counts().index[0]
    X_dataframe[column_name] = X_dataframe[column_name].fillna(most_frequent_value)
    return X_dataframe


def remove_row(X_dataframe: pd.DataFrame, column_name=None, input=None) -> pd.DataFrame:
    indexes = X_dataframe[X_dataframe[column_name].isna()].index.to_list()
    X_dataframe.drop(index=indexes, inplace=True)
    return X_dataframe


"""Functions for data preprocessing"""

"""Functions for categorical data"""

def one_hot_encode_labels(y: pd.Series, categories: list[str]) -> pd.DataFrame:
    new_y_dataframe = pd.DataFrame()
    y = y.apply(lambda y_value: str(y_value))
    for category in categories:
        new_y_dataframe[category] = y == str(category)
        new_y_dataframe[category] = new_y_dataframe[category].astype(int)

    return new_y_dataframe


def one_hot_encode_column(X_dataframe: pd.DataFrame, column_name: str, categories: list[str]):
    for category in categories:
        X_dataframe[category] = X_dataframe[column_name] == category
        X_dataframe[category] = X_dataframe[category].astype(int)
    X_dataframe.drop(column_name, axis=1, inplace=True)
    return X_dataframe


def label_encode_column(X_dataframe: pd.DataFrame, column_name: str, categories: list[str]):
    for category, number in zip(categories, range(len(categories))):
        X_dataframe[category] = X_dataframe[column_name] == category
        X_dataframe[category] = X_dataframe[category].astype(int)
    return X_dataframe


"""Functions for numerical data"""


def min_max_scale_column(X_dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    X_dataframe[column_name] = (X_dataframe[column_name] - X_dataframe[column_name].min()) / (X_dataframe[column_name].max() - X_dataframe[column_name].min())
    return X_dataframe


def std_scale_column(X_dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    column_mean = X_dataframe[column_name].mean()
    column_std = X_dataframe[column_name].std()
    X_dataframe[column_name] = (X_dataframe[column_name] - column_mean) / column_std
    return X_dataframe

**Warning about the former functions**

The former functions rewrite the dataframe as you pass it through them. Therefore, make sure to input null values before the preprocessing. See an example below. As we pass X_train, the column remains transformed.

In [ ]:
print(std_scale_column(X_train, column_name="age"))

In [ ]:
display(X_train.head())

Exercise:

Consider designing two preprocessing pipelines. For simplicity, we will consider a the preprocessing pipeline to be the set of columns with their respective null-value imputation functions and preprocessing functions assigned. For the first, use only null value imputation and preprocessing for categorical data, while for the second introduce preprocessing functions for numerical data.

We will use them later.

Example with the age column:

1st Pipeline:
age - mean_input.
workclass - most_frequent and one_hot_encoder.

2nd Pipeline
age - mean_input and std_scale_column.
workclass - most_frequent and one_hot_encoder.

You may define it in markdown below, or directly code it in python. If you go for the second, name the datasets of the two pipelines differently, to not overwrite the information in each. Do so before the code with the function copy for each partition of the dataset.

In [ ]:
# The model is not prepared for one-hot-encode the labels. If you want to 
# one-hot-encode them, you must change the loss from binary_crossentropy 
# to categorical_crossentropy

y_train = y_train.apply(lambda cell: 1 if cell==" >50K" else 0)
y_val = y_val.apply(lambda cell: 1 if cell==" >50K" else 0)
y_test = y_test.apply(lambda cell: 1 if cell==" >50K" else 0)

y_train = y_train.loc[X_train.index.to_list()]
y_val = y_val.loc[X_val.index.to_list()]
y_test = y_test.loc[X_test.index.to_list()]

## 4. Baseline Model: Logistic Regression with Tensorflow

Based on the following [tutorial](https://www.tensorflow.org/guide/core/logistic_regression_core).

We will not follow fully the tutorial, as it uses the low-level API. Such API is outside the scope of the course. Therefore, we will limit ourselves to the Functional API of Tensorflow, which was transferred to Keras 3.0. Furthermore, as some of you might be familiar with Pytorch, you can still change backend for the training without changing the model definition when using Keras:

In [ ]:
import keras

ml_model: keras.Sequential = keras.Sequential()

Now, we need to add the input layer. This layer must have the same dimensions as the number of columns of the dataset.

In [ ]:
input_parameters = X_train.shape[1]

ml_model.add(keras.Input(shape=(input_parameters,)))

Now, we add the layers in charge of performing the logistic regression

In [ ]:
ml_model.add(keras.layers.Dense(1, activation=keras.activations.sigmoid, kernel_initializer=keras.initializers.Zeros()))

Before performing the training, we must provide the model with a loss function, and compile it.

In [ ]:
import tensorflow as tf

ml_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = ml_model.fit(
    X_train,
    y_train,
    epochs=100,
    # Suppress logging.
    verbose=2,
    # Calculate validation results on 20% of the training data.
    validation_data=(X_val, y_val)
)

Exercise:

Run the logistic regression with the datasets generated by both pipelines.

Is there are difference in performance between them?

What do you think it can be the cause?

## 5. Neural Network Model in Tensorflow.

Let's build now a more complex model to compare it with the former model.

Here, we have defined a 3 layer Multi Layer Perceptron. Each layer is defined in Tensorflow with Dense. Then, the following layer is a Dropout layer, whose functioning is to transform to cero certain inputs from the former layers, in order to improve generalization and performance under missingness. Finally, we apply the same activation (sigmoid), for classification on 

In [ ]:
# --- Architecture hyperparameters (fill these in) ---
layer_1_units      = ???          # e.g. 64
layer_1_activation = ???          # e.g. 'relu'

layer_2_units      = ???          # e.g. 32
layer_2_activation = ???          # e.g. 'relu'

layer_3_units      = ???          # e.g. 16
layer_3_activation = ???          # e.g. 'relu'

dropout_rate       = ???          # e.g. 0.3 (between 0.0 and 1.0)

# --- Model definition ---
mlp_model = keras.Sequential([
    keras.Input(shape=(input_parameters,)),
    keras.layers.Dense(layer_1_units, activation=layer_1_activation),
    keras.layers.Dropout(dropout_rate),
    keras.layers.Dense(layer_2_units, activation=layer_2_activation),
    keras.layers.Dropout(dropout_rate),
    keras.layers.Dense(layer_3_units, activation=layer_3_activation),
    keras.layers.Dropout(dropout_rate),
    keras.layers.Dense(1, activation='sigmoid')
])

mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

mlp_model.summary()

In [ ]:
history = mlp_model.fit(
    X_train,
    y_train,
    epochs=20,
    # Suppress logging.
    verbose=2,
    # Calculate validation results on 20% of the training data.
    validation_data=(X_val, y_val)
)

Exercise:

Run the new model with the datasets generated by both pipelines.

Is there are difference in performance between them?

What do you think it can be the cause?

Do you think that you can design another preprocessing pipeline to increase performance?

## 6. Hyperparameter Optimization with Optuna.

Choosing layer sizes, activation functions, and learning rates by hand can become too much, specially when done manually. [Optuna](https://optuna.org/) is an automatic hyperparameter optimization framework that searches for the best configuration by running many *trials*, each with a different set of hyperparameters.

Each trial is evaluated with an **objective function** that trains the model and returns a score (e.g. validation loss). Optuna uses this score to intelligently propose better hyperparameters for the next trial, using a strategy called **Tree-structured Parzen Estimator (TPE)** by default.

For it, it creates a **study**, where multiple **trials** are run. Each trial is the execution of the training with a specific set of hyperparameters. The results, based on the defined **objective**, are stored and used for optuna to plan for the next trial.

To obtain the values, the function suggest_(type of data) is used, as it can be seen below. 

In [ ]:
import optuna

def objective(trial):

    # --- Hyperparameter search space ---
    layer_1_units      = trial.suggest_int("layer_1_units", 16, 128)
    layer_2_units      = trial.suggest_int("layer_2_units", 16, 128)
    layer_3_units      = trial.suggest_int("layer_3_units", 8, 64)
    activation         = trial.suggest_categorical("activation", ["relu", "tanh", "elu"])
    learning_rate      = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    dropout_rate       = trial.suggest_float("dropout_rate", 0.0, 0.5)

    # --- Build model ---
    model = keras.Sequential([
        keras.Input(shape=(input_parameters,)),
        keras.layers.Dense(layer_1_units, activation=activation),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(layer_2_units, activation=activation),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(layer_3_units, activation=activation),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # --- Train ---
    history = model.fit(
        X_train.astype(np.float32),
        y_train.astype(np.float32),
        epochs=30,
        validation_split=0.2,
        verbose=0
    )

    val_loss = history.history['val_loss'][-1]
    return val_loss


# --- Run the study ---
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best trial:")
print(f"  Val loss:  {study.best_trial.value:.4f}")
print(f"  Params:    {study.best_trial.params}")